In [81]:
pip install -U langchain-google-genai

In [160]:
from google.colab import userdata
import os
from langchain_google_genai import ChatGoogleGenerativeAI
os.environ['GOOGLE_API_KEY']=userdata.get('GemKey2')
model=ChatGoogleGenerativeAI(
    model='gemini-2.5-flash'
)

#### youtube transcription

In [83]:
pip install youtube-transcript-api


In [84]:
from youtube_transcript_api import YouTubeTranscriptApi

In [85]:
## Video id is always present between / and ?
# for Example : id URL is: https://youtu.be/o126p1QN_RI?si=FHTpljtHGvz8IblP

In [86]:
url='https://youtu.be/un0SjUnHvvE?si=ae4Lu4AMWGlBKnnS'
video_id=url.split("/")[-1].split("?")[0]
YT=YouTubeTranscriptApi()
result=YT.fetch(video_id)

In [87]:
video_id

'un0SjUnHvvE'

In [88]:
result

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='hello all my name is krishn and welcome', start=0.399, duration=3.96), FetchedTranscriptSnippet(text='to my YouTube channel so guys uh', start=2.52, duration=3.92), FetchedTranscriptSnippet(text='recently I think everybody has seen the', start=4.359, duration=4.761), FetchedTranscriptSnippet(text='Google IO announcement of 2024 and', start=6.44, duration=5.119), FetchedTranscriptSnippet(text='Google has specifically released many', start=9.12, duration=5.08), FetchedTranscriptSnippet(text='many variants of gemin model uh in this', start=11.559, duration=4.8), FetchedTranscriptSnippet(text="video I'm going to talk about a model", start=14.2, duration=5.48), FetchedTranscriptSnippet(text="which is called as Palama and I hope I'm", start=16.359, duration=5.281), FetchedTranscriptSnippet(text="pronouncing it right we'll be talking", start=19.68, duration=3.48), FetchedTranscriptSnippet(text='about this specific model what this', st

## Direct or Basic menthod (without retriver and generation)
- Already we loaded the data , and no need for text splitter bcoz it is already in form of documents

In [89]:

string=' '
for i in range(len(result)):
  string =string+ ' ' +result[i].text

In [196]:
string

"  hello all my name is krishn and welcome to my YouTube channel so guys uh recently I think everybody has seen the Google IO announcement of 2024 and Google has specifically released many many variants of gemin model uh in this video I'm going to talk about a model which is called as Palama and I hope I'm pronouncing it right we'll be talking about this specific model what this model actually does uh it is also a complete open source model you know and uh we'll also be discussing how we can actually implement it and how we can use it from hugging face okay so all those things I will be specifically discussing in this particular video so to start with uh poama is an open Vision language model and uh this is completely open source uh that basically means you can put up images you can get information from that specific images and all um it is a powerful open VM okay so if you don't know about VM please I would suggest go ahead and read this particular research paper po3 Vision language m

In [197]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate, ChatPromptTemplate
parser=StrOutputParser()

In [202]:
SYS_temp=''' you a an experting in industanding the youtube lectures and
 transcript here your main job is to userstand the below {context}
 and give the detailed notes such that the user can able to easily
  understand, give t the output with side headings and short bullet points,
   anser the user query give answer in one line'''
user_temp='{query}'
sys_msg=SystemMessagePromptTemplate.from_template(SYS_temp)
human_msg=HumanMessagePromptTemplate.from_template(user_temp)
Promptvar=ChatPromptTemplate.from_messages([sys_msg,human_msg])

In [203]:
chain1= Promptvar | model |parser
res=chain1.invoke({
    'context': string,
    'query': ' which latest model that the user were talking about'
})

In [204]:
print(res)

The latest model the user was talking about is **Palama**.


## Using RAG system
- detailed step-by-step
- Vector db

In [134]:
pip install -qU langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 42.0 MB/s eta 0:00:00


In [165]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
os.environ['GOOGLE_API_KEY']=userdata.get('GemKey2')
embeddings = GoogleGenerativeAIEmbeddings(

    model="models/gemini-embedding-001",
    output_dimensionality=768, )

#### Since `result` were in form of FetchedTranscriptSnippet which is not accepted by embedding so we need to convert them into documents inorder to obtain accurate conversion

In [166]:
from langchain_core.documents import Document

In [167]:
docs=[]
for i in range(len(result)):
  docs.append(Document(
    page_content=result[i].text
   ))

In [168]:
  docs[:5]

[Document(metadata={}, page_content='hello all my name is krishn and welcome'),
 Document(metadata={}, page_content='to my YouTube channel so guys uh'),
 Document(metadata={}, page_content='recently I think everybody has seen the'),
 Document(metadata={}, page_content='Google IO announcement of 2024 and'),
 Document(metadata={}, page_content='Google has specifically released many')]

In [170]:

pip install -qU langchain-chroma


In [171]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="youtube",
    embedding_function=embeddings,
)

In [183]:
vector_store.from_documents(
    documents=docs[:90], # Makesure give the lower chunk size then only it will able to coonvert or embbed them
    embedding=embeddings,
    collection_name="youtube",
)

### Retriver

In [184]:
retriver=vector_store.as_retriever(
   search_type='similarity'
)

In [185]:
vector_store.similarity_search('name')

[Document(id='f293d488-dc93-4b6e-8680-8d5dde82f58f', metadata={}, page_content='hello all my name is krishn and welcome'),
 Document(id='6ea4a70e-9657-4506-9e82-4ad0f245f575', metadata={}, page_content='hello all my name is krishn and welcome'),
 Document(id='992b6f80-9644-4cb0-999e-9c7b04f70c5f', metadata={}, page_content='here I will be putting the link in the'),
 Document(id='74b0bce0-bc4b-4f7c-a0a9-e66b4bfe873d', metadata={}, page_content='here I will be putting the link in the')]

### GENERATION

In [186]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate, ChatPromptTemplate
parser=StrOutputParser()
from langchain_core.runnables import RunnablePassthrough

In [187]:
SYS_temp=''' you a an experting in industanding the youtube lectures and
 transcript here your main job is to userstand the below {context}
 and give the detailed notes such that the user can able to easily
  understand, give t the output with side headings and short bullet points,
   anser the user query'''
user_temp='{query}'
sys_msg=SystemMessagePromptTemplate.from_template(SYS_temp)
human_msg=HumanMessagePromptTemplate.from_template(user_temp)
Promptvar=ChatPromptTemplate.from_messages([sys_msg,human_msg])

In [188]:
chain={'context':retriver,
       'query': RunnablePassthrough()}|Promptvar|model|parser

In [180]:
chain.invoke(" what is content about ")

'Based on the provided snippets, the content is about:\n\n### Video Content Overview\n\n*   **Specific Model Discussion:** The core focus will be a detailed explanation or discussion concerning a particular model.\n*   **Video Description:** The content is likely part of a video, and it will include a description of that specific video.\n*   **Detailed Discussion Points:** The speaker or presenter will outline the exact topics or "things" they plan to discuss in detail.\n*   **Additional Information:** More information related to the model or the discussion will be provided.'

In [193]:
useerinput=input("Hello iam Youtube transcripter ..Please ASK question")
print(chain.invoke(useerinput))

Hello iam Youtube transcripter ..Please ASK questionName of auther
Based on the provided document snippets, the author's name is:

### Author Information

*   **Name:** Krishn


In [192]:
useerinput=input("name the model which auther is discusssing about")
print(chain.invoke(useerinput)) # Not giving bcoz we entered limited chunks of data

name the model which auther is discusssing aboutname the model which auther is discusssing about
Based on the provided text snippets, the specific name of the model being discussed by the author is **not mentioned**.

Here are the detailed notes from the documents:

### Model Identification

*   The text frequently refers to "a model" or "this particular model."
*   However, no specific name or descriptive term for the model is provided within these snippets.

### Information Provided About the Model

*   **Discussion Topic:** The author intends to "talk about a model" in the video.
*   **Specificity:** It's referred to as "this particular model itself," indicating it's a specific subject of the discussion, even if its name isn't given.
*   **Interaction:** There's a mention of "model okay so once you probably click on," which might suggest an interactive element or a step related to the model.

### Conclusion

*   The provided text **does not contain the name** of the model the author

In [191]:
docs[:90]

[Document(metadata={}, page_content='hello all my name is krishn and welcome'),
 Document(metadata={}, page_content='to my YouTube channel so guys uh'),
 Document(metadata={}, page_content='recently I think everybody has seen the'),
 Document(metadata={}, page_content='Google IO announcement of 2024 and'),
 Document(metadata={}, page_content='Google has specifically released many'),
 Document(metadata={}, page_content='many variants of gemin model uh in this'),
 Document(metadata={}, page_content="video I'm going to talk about a model"),
 Document(metadata={}, page_content="which is called as Palama and I hope I'm"),
 Document(metadata={}, page_content="pronouncing it right we'll be talking"),
 Document(metadata={}, page_content='about this specific model what this'),
 Document(metadata={}, page_content='model actually does uh it is also a'),
 Document(metadata={}, page_content='complete open source model you know and'),
 Document(metadata={}, page_content="uh we'll also be discussing 